<a href="https://colab.research.google.com/github/debashisdotchatterjee/Bird-Flu-Epidemic-1/blob/main/Bird_Flu_Epidemic_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***MAIN CODE WHOLE ***

In [ ]:
# A small fix for day-of-year extraction from numpy.datetime64 dates

import os
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import pi
from datetime import datetime

try:
    from scipy.optimize import minimize
    SCIPY_OK = True
except Exception as e:
    SCIPY_OK = False

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
        for col in df.columns:
            if col.lower() == c.lower():
                return col
    return None

def to_datetime_series(s):
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.tz_localize(None)
    return pd.to_datetime(s, errors="coerce").dt.tz_localize(None)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arcsin(np.minimum(1.0, np.sqrt(a)))
    return R*c

def discrete_gamma_weights(L=14, mean=7.0, sd=3.0):
    k = (mean/sd)**2 if sd>0 else 1.0
    theta = (sd**2)/mean if mean>0 else 1.0
    from math import gamma as Gamma
    t = np.arange(1, L+1, dtype=float)
    pdf = (t**(k-1)) * np.exp(-t/theta) / (Gamma(k) * (theta**k))
    w = pdf / pdf.sum()
    return w

def nb2_nll(y, mu, psi):
    mu = np.clip(mu, 1e-9, None)
    psi = max(psi, 1e-6)
    from scipy.special import gammaln
    r = psi
    term = (gammaln(y + r) - gammaln(r) - gammaln(y + 1)
            + r*np.log(r/(r+mu)) + y*np.log(mu/(r+mu)))
    return -np.sum(term)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

csv_path = "data-table.csv"
assert os.path.exists(csv_path), "CSV not found at data-table.csv"
raw = pd.read_csv(csv_path)

date_col = find_col(raw, ["date","Date","report_date","confirmation_date","Reported Date","event_date"])
state_col = find_col(raw, ["state","State","province_state","Province_State"])
county_col = find_col(raw, ["county","County","county_name","County_Name"])
lat_col = find_col(raw, ["lat","latitude","Lat","Latitude"])
lon_col = find_col(raw, ["lon","long","longitude","Longitude","Long"])

if date_col is None:
    alt = find_col(raw, ["day","Day","week","Week"])
    if alt is not None:
        raw["__date__"] = pd.to_datetime(raw[alt], errors="coerce")
        date_col = "__date__"
    else:
        raw["__date__"] = pd.to_datetime("2022-01-01") + pd.to_timedelta(np.arange(len(raw)), unit="D")
        date_col = "__date__"

if state_col is None:
    raw["__state__"] = "Unknown"
    state_col = "__state__"

if county_col is None:
    raw["__county__"] = raw[state_col].astype(str)
    county_col = "__county__"

raw[date_col] = to_datetime_series(raw[date_col])
raw = raw.dropna(subset=[date_col])

count_col = find_col(raw, ["count","detections","n","events","Cases","cases"])
if count_col is None:
    raw["__count__"] = 1
    count_col = "__count__"

raw = raw[raw[date_col] >= pd.Timestamp("2022-01-01")].copy()

raw[state_col] = raw[state_col].astype(str).str.strip()
raw[county_col] = raw[county_col].astype(str).str.strip()
raw["county_id"] = raw[state_col] + "|" + raw[county_col]

county_meta = raw.groupby("county_id").agg(
    state=(state_col, "first"),
    county=(county_col, "first"),
    lat=(lat_col, "mean") if lat_col else ("county_id","count"),
    lon=(lon_col, "mean") if lon_col else ("county_id","count")
).reset_index()

date_min = raw[date_col].min().normalize()
date_max = raw[date_col].max().normalize()
all_days = pd.date_range(date_min, date_max, freq="D")

daily = (raw
         .groupby(["county_id", pd.Grouper(key=date_col, freq="D")])[count_col]
         .sum()
         .rename("y")
         .reset_index()
         .rename(columns={date_col: "date"}))

counties = county_meta["county_id"].tolist()
grid = (pd.MultiIndex.from_product([counties, all_days], names=["county_id","date"])
        .to_frame(index=False))
panel = grid.merge(daily, on=["county_id","date"], how="left").fillna({"y":0})
panel = panel.merge(county_meta[["county_id","state","county"]], on="county_id", how="left")

if lat_col and lon_col and pd.api.types.is_numeric_dtype(county_meta["lat"]):
    panel = panel.merge(county_meta[["county_id","lat","lon"]], on="county_id", how="left")
else:
    panel["lat"] = np.nan
    panel["lon"] = np.nan

unique_counties = county_meta["county_id"].tolist()
n = len(unique_counties)

MAX_COUNTIES = 400
if n > MAX_COUNTIES:
    totals = panel.groupby("county_id")["y"].sum().sort_values(ascending=False)
    keep = set(totals.head(MAX_COUNTIES).index.tolist())
    panel = panel[panel["county_id"].isin(keep)].copy()
    county_meta = county_meta[county_meta["county_id"].isin(keep)].copy()
    unique_counties = county_meta["county_id"].tolist()
    n = len(unique_counties)

county_index = {cid:i for i, cid in enumerate(unique_counties)}
panel["i"] = panel["county_id"].map(county_index)
states = county_meta.set_index("county_id")["state"].to_dict()

lat_avail = county_meta["lat"].notna().all()
if lat_avail:
    lats = county_meta.set_index("county_id").loc[unique_counties, "lat"].values
    lons = county_meta.set_index("county_id").loc[unique_counties, "lon"].values
    Di = np.zeros((n,n), dtype=float)
    for i in range(n):
        Di[i,:] = haversine_km(lats[i], lons[i], lats, lons)
else:
    Di = np.full((n,n), 1000.0, dtype=float)
    states_arr = county_meta.set_index("county_id").loc[unique_counties, "state"].values
    for i in range(n):
        for j in range(n):
            if states_arr[i] == states_arr[j]:
                Di[i,j] = 50.0
    np.fill_diagonal(Di, 0.0)

L = 14
w = discrete_gamma_weights(L=L, mean=7.0, sd=3.0)

dates = np.sort(panel["date"].unique())
T = len(dates)
panel = panel.sort_values(["i","date"])

Y = np.zeros((n, T), dtype=float)
for cid, sub in panel.groupby("i"):
    Y[cid, :] = sub["y"].values

H = np.zeros_like(Y)
for j in range(n):
    series = Y[j,:]
    conv = np.convolve(series, w, mode="full")[:T]
    conv_shift = np.concatenate([[0.0], conv[:-1]])
    H[j,:] = conv_shift

# FIX: use pandas DatetimeIndex to get dayofyear
day_of_year = pd.DatetimeIndex(dates).dayofyear.values.astype(float)
omega = 2.0 * np.pi / 365.25
sin_seas = np.sin(omega*day_of_year)
cos_seas = np.cos(omega*day_of_year)

SIN = np.tile(sin_seas, (n,1))
COS = np.tile(cos_seas, (n,1))

def build_kernel(D, rho=1.5, d0=10.0):
    K = (D + d0)**(-rho)
    row_sums = K.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return K / row_sums

rho0 = 1.5
d0 = 10.0
K = build_kernel(Di, rho=rho0, d0=d0)
E = K @ H

state_list = sorted(panel.drop_duplicates("county_id").set_index("county_id")["state"].unique().tolist())
S = len(state_list)
state_to_idx = {s:i for i,s in enumerate(state_list)}
county_state_idx = np.array([state_to_idx[states[cid]] for cid in unique_counties], dtype=int)

t0 = L
fit_mask = np.zeros((n,T), dtype=bool)
fit_mask[:, t0:] = True

y_vec = Y[fit_mask]
sin_vec = SIN[fit_mask]
cos_vec = COS[fit_mask]
E_vec = E[fit_mask]

county_idx_grid = np.repeat(np.arange(n)[:,None], T, axis=1)
state_idx_grid = np.vectorize(lambda i: county_state_idx[i])(county_idx_grid)
state_idx_vec = state_idx_grid[fit_mask]

def unpack_theta(theta, S, estimate_rho):
    a0 = theta[0]
    s1 = theta[1]
    s2 = theta[2]
    b0 = theta[3]
    c1 = theta[4]
    c2 = theta[5]
    log_psi = theta[6]
    idx = 7
    if estimate_rho:
        log_rho = theta[idx]; idx += 1
        rho = np.exp(log_rho)
    else:
        rho = None
    state_FE = np.zeros(S, dtype=float)
    state_FE[:-1] = theta[idx: idx + (S-1)]
    return a0, s1, s2, b0, c1, c2, np.exp(log_psi), rho, state_FE

def build_lambda(theta, estimate_rho=False):
    a0, s1, s2, b0, c1, c2, psi, rho, state_FE = unpack_theta(theta, S, estimate_rho)
    if estimate_rho:
        K2 = build_kernel(Di, rho=rho, d0=d0)
        E2 = K2 @ H
        E_local = E2[fit_mask]
    else:
        E_local = E_vec
    fe_by_county = state_FE[county_state_idx]
    FE = np.tile(fe_by_county[:,None], (1,T))[fit_mask]
    nu = np.exp(a0 + s1*sin_vec + s2*cos_vec + FE)
    phi_t = np.exp(b0 + c1*sin_vec + c2*cos_vec)
    lam = nu + phi_t * E_local
    return lam, psi

def objective(theta, estimate_rho=False):
    lam, psi = build_lambda(theta, estimate_rho=estimate_rho)
    return nb2_nll(y_vec, lam, psi)

a0_0 = np.log(max(np.mean(y_vec)+1e-6, 1e-6))
init = [a0_0, 0.0, 0.0, np.log(0.1 + (np.mean(y_vec) if np.mean(y_vec)>0 else 0.1)), 0.0, 0.0, np.log(10.0)]
estimate_rho = lat_avail
if estimate_rho:
    init.append(np.log(rho0))
init.extend([0.0]*(S-1))
init = np.array(init, dtype=float)

bounds = []
for _ in range(6):
    bounds.append((None, None))
bounds.append((np.log(1e-3), np.log(1e6)))
if estimate_rho:
    bounds.append((np.log(0.1), np.log(5.0)))
bounds.extend([(None,None)]*(S-1))

if SCIPY_OK:
    res = minimize(objective, init, args=(estimate_rho,), method="L-BFGS-B", bounds=bounds, options={"maxiter": 300, "ftol":1e-6})
    theta_hat = res.x
    success = res.success
    msg = res.message
    nll = res.fun
else:
    theta_hat = init.copy()
    success = False
    msg = "SciPy not available; using initial parameters (no optimization)."
    nll = objective(theta_hat, estimate_rho=estimate_rho)

lam_hat, psi_hat = build_lambda(theta_hat, estimate_rho=estimate_rho)
a0, s1, s2, b0, c1, c2, psi_val, rho_val, state_FE = unpack_theta(theta_hat, S, estimate_rho)

if estimate_rho:
    K = build_kernel(Di, rho=rho_val, d0=d0)
    E = K @ H

fe_by_county = state_FE[county_state_idx]
FE_full = np.tile(fe_by_county[:,None], (1,T))
NU_full = np.exp(a0 + s1* np.tile(sin_seas, (n,1)) + s2* np.tile(cos_seas, (n,1)) + FE_full)
PHI_full = np.exp(b0 + c1* np.tile(sin_seas, (n,1)) + c2* np.tile(cos_seas, (n,1)))
LAM_full = NU_full + PHI_full * E

outdir = "h5n1_outputs"
os.makedirs(outdir, exist_ok=True)

summary = {
    "n_rows_raw": [len(raw)],
    "n_uniq_states": [panel["state"].nunique()],
    "n_uniq_counties_used": [n],
    "date_min": [str(pd.Timestamp(date_min).date())],
    "date_max": [str(pd.Timestamp(date_max).date())],
    "total_detections": [int(panel["y"].sum())],
    "scipy_optimization_success": [bool(success)],
    "optimizer_message": [str(msg)],
    "negloglik": [float(nll)],
    "psi_hat": [float(psi_val)],
}
if estimate_rho:
    summary["rho_hat"] = [float(rho_val)]
summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(outdir, "summary.csv"), index=False)

state_totals = (panel.groupby("state")["y"].sum()
                .sort_values(ascending=False)
                .reset_index()
                .rename(columns={"y":"total_detections"}))
state_totals.to_csv(os.path.join(outdir, "state_totals.csv"), index=False)

AF_end = NU_full / np.maximum(LAM_full, 1e-9)
AF_epi = 1.0 - AF_end
attr_by_day = pd.DataFrame({
    "date": pd.DatetimeIndex(dates),
    "expected_total": LAM_full.sum(axis=0),
    "endemic_expected": NU_full.sum(axis=0),
    "epidemic_expected": (PHI_full*E).sum(axis=0),
    "AF_endemic": AF_end.mean(axis=0),
    "AF_epidemic": AF_epi.mean(axis=0),
})
attr_by_day.to_csv(os.path.join(outdir, "attribution_by_day.csv"), index=False)

state_idx_per_county = county_state_idx
state_names = state_list
attr_state_rows = []
for si, sname in enumerate(state_names):
    idxs = np.where(state_idx_per_county == si)[0]
    if len(idxs)==0:
        continue
    lam_s = LAM_full[idxs,:].sum()
    end_s = NU_full[idxs,:].sum()
    epi_s = (PHI_full[idxs,:]*E[idxs,:]).sum()
    attr_state_rows.append({
        "state": sname,
        "expected_total": lam_s,
        "endemic_expected": end_s,
        "epidemic_expected": epi_s,
        "AF_endemic": end_s / max(lam_s, 1e-9),
        "AF_epidemic": epi_s / max(lam_s, 1e-9),
    })
attr_state = pd.DataFrame(attr_state_rows).sort_values("expected_total", ascending=False)
attr_state.to_csv(os.path.join(outdir, "attribution_by_state.csv"), index=False)

param_rows = [
    {"parameter":"a0 (endemic intercept)", "estimate": a0},
    {"parameter":"s1 (endemic sin)", "estimate": s1},
    {"parameter":"s2 (endemic cos)", "estimate": s2},
    {"parameter":"b0 (log phi intercept)", "estimate": b0},
    {"parameter":"c1 (phi sin)", "estimate": c1},
    {"parameter":"c2 (phi cos)", "estimate": c2},
    {"parameter":"psi (NB dispersion)", "estimate": psi_val},
]
if estimate_rho:
    param_rows.append({"parameter":"rho (kernel exponent)", "estimate": rho_val})
for i, sname in enumerate(state_list[:-1]):
    param_rows.append({"parameter": f"state_FE[{sname}] (ref={state_list[-1]})", "estimate": state_FE[i]})
param_df = pd.DataFrame(param_rows)
param_df.to_csv(os.path.join(outdir, "parameter_estimates.csv"), index=False)

plt.rcParams.update({"figure.dpi": 120})

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()

F1 = os.path.join(outdir, "fig1_national_daily_detections.png")
ts_nat = panel.groupby("date")["y"].sum().reindex(pd.DatetimeIndex(dates)).fillna(0)
plt.figure()
plt.plot(ts_nat.index, ts_nat.values, lw=2.0, color="#1f77b4", label="Observed detections (US)")
plt.xlabel("Date")
plt.ylabel("Detections per day")
plt.title("USDA H5N1 Poultry Detections — Daily (Observed)")
plt.legend()
savefig(F1)

F2 = os.path.join(outdir, "fig2_top_states_bar.png")
top_states = state_totals.head(15)
plt.figure()
plt.bar(top_states["state"], top_states["total_detections"], color="#d62728")
plt.xticks(rotation=60, ha="right")
plt.ylabel("Total detections")
plt.title("Top 15 States by Total H5N1 Detections")
savefig(F2)

F3 = os.path.join(outdir, "fig3_state_week_heatmap.png")
panel["week"] = panel["date"].dt.to_period("W").dt.start_time
state_week = panel.groupby(["state","week"])["y"].sum().reset_index()
heat = state_week.pivot(index="state", columns="week", values="y").fillna(0)
heat = heat.loc[state_totals["state"]]
plt.figure()
plt.imshow(heat.values, aspect="auto", interpolation="nearest")
plt.colorbar(label="Detections per week")
plt.yticks(np.arange(len(heat.index)), heat.index)
xt = np.linspace(0, heat.shape[1]-1, num=min(12, heat.shape[1])).astype(int)
plt.xticks(xt, [str(pd.Timestamp(heat.columns[i]).date()) for i in xt], rotation=60, ha="right")
plt.title("Weekly H5N1 Detections by State (Heatmap)")
savefig(F3)

F4 = os.path.join(outdir, "fig4_seasonality.png")
df_seas = pd.DataFrame({"doy": pd.DatetimeIndex(dates).dayofyear, "y": ts_nat.values})
seas_mean = df_seas.groupby("doy")["y"].mean()
plt.figure()
plt.plot(seas_mean.index, seas_mean.values, lw=2.0, color="#2ca02c")
plt.xlabel("Day of Year")
plt.ylabel("Mean detections")
plt.title("Seasonal Pattern (Mean by Day-of-Year)")
savefig(F4)

if lat_avail:
    F5 = os.path.join(outdir, "fig5_kernel_vs_distance.png")
    dvals = np.linspace(0, max(1.0, np.nanmax(Di)), 200)
    kvals = (dvals + 10.0)**(- (rho_val if SCIPY_OK and len(init)>0 and ('rho_hat' in locals() or True) else rho0))
    kvals = kvals / np.sum(kvals)
    plt.figure()
    plt.plot(dvals, kvals, lw=2.0, color="#9467bd")
    plt.xlabel("Distance (km)")
    plt.ylabel("Kernel weight (scaled)")
    plt.title("Power-law Kernel vs Distance")
    savefig(F5)

F6 = os.path.join(outdir, "fig6_endemic_vs_epidemic.png")
endemic_nat = NU_full.sum(axis=0)
epidemic_nat = (PHI_full*E).sum(axis=0)
plt.figure()
plt.fill_between(pd.DatetimeIndex(dates), endemic_nat, color="#17becf", alpha=0.6, label="Endemic (importation) expected")
plt.plot(pd.DatetimeIndex(dates), epidemic_nat, lw=2.0, color="#ff7f0e", label="Epidemic (spread) expected")
plt.xlabel("Date")
plt.ylabel("Expected detections")
plt.title("Endemic vs Epidemic Contributions (National)")
plt.legend()
savefig(F6)

F7 = os.path.join(outdir, "fig7_obs_vs_fit.png")
lam_nat = LAM_full.sum(axis=0)
plt.figure()
plt.plot(pd.DatetimeIndex(dates), ts_nat.values, lw=1.5, color="#1f77b4", label="Observed")
plt.plot(pd.DatetimeIndex(dates), lam_nat, lw=2.0, color="#d62728", label="Fitted (expected)")
plt.xlabel("Date")
plt.ylabel("Detections per day")
plt.title("Observed vs Fitted (National)")
plt.legend()
savefig(F7)

F8 = os.path.join(outdir, "fig8_county_totals_hist.png")
county_totals = panel.groupby("county_id")["y"].sum()
plt.figure()
plt.hist(county_totals.values, bins=30, color="#8c564b")
plt.xlabel("Total detections per county")
plt.ylabel("Number of counties")
plt.title("Distribution of County Totals")
savefig(F8)

eps = 1e-9
obs = ts_nat.values[t0:]
fit_nat = lam_nat[t0:]
mae = np.mean(np.abs(obs - fit_nat))
rmse = float(np.sqrt(np.mean((obs - fit_nat)**2)))
log_score = float(np.mean(np.log(fit_nat + eps) - fit_nat))
metrics = pd.DataFrame([{"MAE_nat": mae, "RMSE_nat": rmse, "Crude_logscore_like": log_score}])
metrics.to_csv(os.path.join(outdir, "metrics.csv"), index=False)

# from caas_jupyter_tools import display_dataframe_to_user
print("USDA H5N1 — Dataset Summary")
display(summary_df)
print("USDA H5N1 — Top States by Detections")
display(state_totals.head(25))
print("USDA H5N1 — Parameter Estimates")
display(param_df)
print("USDA H5N1 — Attribution by Day")
display(attr_by_day.head(30))
print("USDA H5N1 — Attribution by State")
display(attr_state)

zip_path = "h5n1_outputs.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(outdir):
        for f in files:
            full = os.path.join(root, f)
            arc = os.path.relpath(full, start=os.path.dirname(outdir))
            zf.write(full, arc)

print("All outputs saved to:", outdir)
print("ZIP archive created at:", zip_path)

USDA H5N1 — Dataset Summary


,n_rows_raw,n_uniq_states,n_uniq_counties_used,date_min,date_max,total_detections,scipy_optimization_success,optimizer_message,negloglik,psi_hat,rho_hat
0,685,51,51,2022-01-01,2023-11-16,685,True,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,3104.335894,257.024626,2.329267


USDA H5N1 — Top States by Detections


,state,total_detections
0,Minnesota,51.0
1,Iowa,33.0
2,California,30.0
3,South Dakota,30.0
4,Missouri,29.0
5,Michigan,26.0
6,Kansas,26.0
7,Pennsylvania,25.0
8,Wisconsin,23.0
9,Nebraska,23.0


USDA H5N1 — Parameter Estimates


...

USDA H5N1 — Attribution by Day


,date,expected_total,endemic_expected,epidemic_expected,AF_endemic,AF_epidemic
0,2022-01-01,0.857170,0.857170,0.000000,1.000000,0.000000
1,2022-01-02,0.856717,0.856367,0.000349,0.999801,0.000199
2,2022-01-03,0.859459,0.855579,0.003880,0.997842,0.002158
3,2022-01-04,0.868639,0.854806,0.013833,0.992595,0.007405
4,2022-01-05,0.884525,0.854047,0.030478,0.983851,0.016149
5,2022-01-06,0.904706,0.853304,0.051402,0.968072,0.031928
6,2022-01-07,0.925961,0.852576,0.073386,0.950365,0.049635
7,2022-01-08,0.945685,0.851863,0.093822,0.937436,0.062564
8,2022-01-09,0.962397,0.851166,0.111231,0.927967,0.072033
9,2022-01-10,0.975635,0.850484,0.125151,0.916369,0.083631


USDA H5N1 — Attribution by State


,state,expected_total,endemic_expected,epidemic_expected,AF_endemic,AF_epidemic
22,Minnesota,51.272333,45.617497,5.654836,0.889710,0.110290
14,Iowa,33.448688,30.927938,2.520750,0.924638,0.075362
24,Missouri,29.416011,26.898605,2.517406,0.914421,0.085579
41,South Dakota,28.738301,25.558562,3.179738,0.889355,0.110645
4,California,27.250285,24.070547,3.179738,0.883314,0.116686
15,Kansas,25.678353,22.885567,2.792786,0.891240,0.108760
37,Pennsylvania,24.835397,23.018298,1.817100,0.926834,0.073166
26,Nebraska,23.802540,21.519812,2.282728,0.904097,0.095903
49,Wisconsin,23.739578,21.456850,2.282728,0.903843,0.096157
21,Michigan,23.241650,20.448864,2.792786,0.879837,0.120163


All outputs saved to: h5n1_outputs
ZIP archive created at: h5n1_outputs.zip


# **Fast Code**

In [ ]:
# Ultra-fast plug-in decomposition (no heavy optimization):
# - Build weekly node-time panel (states if too many counties)
# - Compute spatial exposure E with fixed kernel (rho=1.5)
# - Fit national endemic baseline nu_t via OLS on (1, sin, cos)
# - Allocate nu_{i,t} across nodes by baseline weights
# - Compute phi_t (time-varying) by matching totals: phi_t = max(0, (y_tot - nu_tot)/E_tot)
# - Produce plots & tables; save ZIP

import os, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
        for col in df.columns:
            if col.lower() == c.lower():
                return col
    return None

def to_datetime_series(s):
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.tz_localize(None)
    return pd.to_datetime(s, errors="coerce").dt.tz_localize(None)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.deg2rad(lat1); lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2); lon2 = np.deg2rad(lon2)
    dlat = lat2 - lat1; dlon = lat2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arcsin(np.minimum(1.0, np.sqrt(a)))
    return R*c

def discrete_gamma_weights(L=6, mean=2.0, sd=1.0):
    from math import gamma as Gamma
    k = (mean/sd)**2 if sd>0 else 1.0
    theta = (sd**2)/mean if mean>0 else 1.0
    t = np.arange(1, L+1, dtype=float)
    pdf = (t**(k-1)) * np.exp(-t/theta) / (Gamma(k) * (theta**k))
    return pdf / pdf.sum()

def build_kernel(D, rho=1.5, d0=10.0):
    K = (D + d0)**(-rho)
    row_sums = K.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return K / row_sums

# Load CSV
csv_path = "data-table.csv"
raw = pd.read_csv(csv_path)

date_col = find_col(raw, ["date","Date","report_date","confirmation_date","Reported Date","event_date"])
state_col = find_col(raw, ["state","State","province_state","Province_State"])
county_col = find_col(raw, ["county","County","county_name","County_Name"])
lat_col = find_col(raw, ["lat","latitude","Lat","Latitude"])
lon_col = find_col(raw, ["lon","long","longitude","Longitude","Long"])

if date_col is None:
    raw["__date__"] = pd.to_datetime("2022-01-01") + pd.to_timedelta(np.arange(len(raw)), unit="D")
    date_col = "__date__"
raw[date_col] = to_datetime_series(raw[date_col])
raw = raw.dropna(subset=[date_col])

if state_col is None:
    raw["__state__"] = "Unknown"
    state_col = "__state__"
if county_col is None:
    raw["__county__"] = raw[state_col].astype(str)
    county_col = "__county__"

count_col = find_col(raw, ["count","detections","n","events","Cases","cases"])
if count_col is None:
    raw["__count__"] = 1
    count_col = "__count__"

raw = raw[raw[date_col] >= pd.Timestamp("2022-01-01")].copy()
raw[state_col] = raw[state_col].astype(str).str.strip()
raw[county_col] = raw[county_col].astype(str).str.strip()
raw["county_id"] = raw[state_col] + "|" + raw[county_col]

# County daily for plots
date_min = raw[date_col].min().normalize(); date_max = raw[date_col].max().normalize()
all_days = pd.date_range(date_min, date_max, freq="D")
daily = (raw.groupby(["county_id", pd.Grouper(key=date_col, freq="D")])[count_col]
         .sum().rename("y").reset_index().rename(columns={date_col:"date"}))
county_meta = raw.groupby("county_id").agg(
    state=(state_col,"first"),
    county=(county_col,"first"),
    lat=(lat_col,"mean") if lat_col else ("county_id","count"),
    lon=(lon_col,"mean") if lon_col else ("county_id","count"),
).reset_index()
grid = pd.MultiIndex.from_product([county_meta["county_id"], all_days], names=["county_id","date"]).to_frame(index=False)
panel = grid.merge(daily, on=["county_id","date"], how="left").fillna({"y":0})
panel = panel.merge(county_meta[["county_id","state","county"]], on="county_id", how="left")

# Use states if too many counties
use_states = len(county_meta) > 120

if use_states:
    panel["week"] = panel["date"].dt.to_period("W").dt.start_time
    state_week = panel.groupby(["state","week"])["y"].sum().reset_index()
    nodes = sorted(state_week["state"].unique().tolist())
    dates_w = pd.date_range(state_week["week"].min(), state_week["week"].max(), freq="W-MON")
    m, Tw = len(nodes), len(dates_w)
    Y = np.zeros((m, Tw))
    node_to_idx = {s:i for i,s in enumerate(nodes)}
    for _, r in state_week.iterrows():
        i = node_to_idx[r["state"]]
        t = np.where(dates_w == pd.Timestamp(r["week"]))[0]
        if len(t)>0: Y[i,t[0]] = r["y"]
    lat_avail = (lat_col is not None) and (lon_col is not None) and pd.api.types.is_numeric_dtype(county_meta["lat"])
    if lat_avail:
        state_ll = panel.groupby("state")[["lat","lon"]].mean().reindex(nodes)
        lats = state_ll["lat"].values; lons = state_ll["lon"].values
        D = np.zeros((m,m))
        for i in range(m): D[i,:] = haversine_km(lats[i], lons[i], lats, lons)
    else:
        D = np.full((m,m), 600.0); np.fill_diagonal(D, 0.0)
else:
    panel["week"] = panel["date"].dt.to_period("W").dt.start_time
    county_week = panel.groupby(["county_id","state","week"])["y"].sum().reset_index()
    nodes = sorted(county_week["county_id"].unique().tolist())
    dates_w = pd.date_range(county_week["week"].min(), county_week["week"].max(), freq="W-MON")
    m, Tw = len(nodes), len(dates_w)
    Y = np.zeros((m, Tw))
    node_to_idx = {cid:i for i,cid in enumerate(nodes)}
    for _, r in county_week.iterrows():
        i = node_to_idx[r["county_id"]]
        t = np.where(dates_w == pd.Timestamp(r["week"]))[0]
        if len(t)>0: Y[i,t[0]] = r["y"]
    cm = county_meta.set_index("county_id").loc[nodes]
    lat_avail = (lat_col is not None) and (lon_col is not None) and pd.api.types.is_numeric_dtype(cm["lat"])
    if lat_avail:
        lats = cm["lat"].values; lons = cm["lon"].values
        D = np.zeros((m,m))
        for i in range(m): D[i,:] = haversine_km(lats[i], lons[i], lats, lons)
    else:
        states_arr = cm["state"].values
        D = np.full((m,m), 800.0)
        for i in range(m):
            for j in range(m):
                if states_arr[i]==states_arr[j]: D[i,j]=60.0
        np.fill_diagonal(D, 0.0)

# Renewal kernel (weekly)
w = discrete_gamma_weights(L=6, mean=2.0, sd=1.0)

# Past pressure H
H = np.zeros_like(Y)
for j in range(m):
    conv = np.convolve(Y[j,:], w, mode="full")[:Tw]
    H[j,:] = np.concatenate([[0.0], conv[:-1]])

# Spatial kernel and exposure
rho0, d0 = 1.5, 10.0
K = build_kernel(D, rho=rho0, d0=d0)
E = K @ H  # m x Tw

# NATIONAL endemic baseline via OLS on (1, sin, cos)
doy = pd.DatetimeIndex(dates_w).dayofyear.values.astype(float)
omega = 2*np.pi/365.25
X = np.column_stack([np.ones(Tw), np.sin(omega*doy), np.cos(omega*doy)])
y_nat = Y.sum(axis=0)
# Least squares
beta, *_ = np.linalg.lstsq(X, y_nat, rcond=None)
nu_nat = X @ beta
nu_nat = np.clip(nu_nat, 1e-6, None)  # enforce positivity

# Allocate endemic to nodes by baseline weights
w_node = Y.mean(axis=1); w_node = w_node / (w_node.sum() if w_node.sum()>0 else 1.0)
NU = np.outer(w_node, nu_nat)  # m x Tw

# Time-varying phi_t by matching totals
E_nat = E.sum(axis=0)
phi_t = (y_nat - nu_nat) / np.maximum(E_nat, 1e-9)
phi_t = np.clip(phi_t, 0.0, None)
# Smooth phi with small moving average (window=3)
if Tw >= 3:
    phi_t = np.convolve(phi_t, np.ones(3)/3.0, mode="same")

# Expected LAM and components
PHI = np.tile(phi_t, (m,1))
LAM = NU + PHI * E

# Outputs
outdir = "h5n1_outputs_ultrafast"
os.makedirs(outdir, exist_ok=True)

# Save tables
summary = pd.DataFrame([{
    "model_level": "state" if use_states else "county",
    "time_scale": "weekly",
    "nodes": int(m),
    "weeks": int(Tw),
    "rho_used": rho0,
    "estimation": "plug-in (OLS seasonality + phi_t by totals matching)",
}])
summary.to_csv(os.path.join(outdir,"summary.csv"), index=False)

attr_by_week = pd.DataFrame({
    "week": dates_w,
    "observed_total": y_nat,
    "expected_total": LAM.sum(axis=0),
    "endemic_expected": NU.sum(axis=0),
    "epidemic_expected": (PHI * E).sum(axis=0),
})
attr_by_week["AF_endemic"] = attr_by_week["endemic_expected"] / np.maximum(attr_by_week["expected_total"], 1e-9)
attr_by_week["AF_epidemic"] = 1.0 - attr_by_week["AF_endemic"]
attr_by_week.to_csv(os.path.join(outdir,"attribution_by_week.csv"), index=False)

node_totals = pd.DataFrame({"node": nodes, "total_detections": Y.sum(axis=1)}).sort_values("total_detections", ascending=False)
node_totals.to_csv(os.path.join(outdir,"node_totals.csv"), index=False)

phi_table = pd.DataFrame({"week": dates_w, "phi_t": phi_t})
phi_table.to_csv(os.path.join(outdir,"phi_t_timeseries.csv"), index=False)

# Plots
plt.rcParams.update({"figure.dpi": 120})

# Daily observed plot
daily_nat = panel.groupby("date")["y"].sum().reindex(pd.date_range(date_min, date_max, freq="D")).fillna(0)
F1 = os.path.join(outdir, "fig1_daily_observed.png")
plt.figure(); plt.plot(daily_nat.index, daily_nat.values, lw=2.0, color="#1f77b4")
plt.title("USDA H5N1 Poultry Detections — Daily (Observed)"); plt.xlabel("Date"); plt.ylabel("Detections/day")
plt.tight_layout(); plt.savefig(F1, bbox_inches="tight"); plt.close()

# Weekly obs vs fitted
F2 = os.path.join(outdir, "fig2_weekly_obs_vs_fit.png")
plt.figure(); plt.plot(dates_w, y_nat, lw=2.0, color="#ff7f0e", label="Observed (weekly)")
plt.plot(dates_w, LAM.sum(axis=0), lw=2.0, color="#2ca02c", label="Fitted (expected)")
plt.title(f"Observed vs Fitted — {'States' if use_states else 'Counties'} (Weekly)"); plt.xlabel("Week"); plt.ylabel("Detections/week"); plt.legend()
plt.tight_layout(); plt.savefig(F2, bbox_inches="tight"); plt.close()

# Endemic vs Epidemic
F3 = os.path.join(outdir, "fig3_endemic_vs_epidemic_weekly.png")
plt.figure(); plt.fill_between(dates_w, NU.sum(axis=0), color="#17becf", alpha=0.6, label="Endemic expected")
plt.plot(dates_w, (PHI*E).sum(axis=0), lw=2.0, color="#d62728", label="Epidemic expected")
plt.title("Endemic vs Epidemic Contributions (Weekly)"); plt.xlabel("Week"); plt.ylabel("Expected detections"); plt.legend()
plt.tight_layout(); plt.savefig(F3, bbox_inches="tight"); plt.close()

# Kernel curve
F4 = os.path.join(outdir, "fig4_kernel_curve.png")
dvals = np.linspace(0, max(1.0, float(np.nanmax(D))), 200)
kvals = (dvals + 10.0)**(-rho0); kvals = kvals / kvals.sum()
plt.figure(); plt.plot(dvals, kvals, lw=2.0, color="#9467bd"); plt.title("Power-law Kernel vs Distance")
plt.xlabel("Distance (km)"); plt.ylabel("Scaled weight"); plt.tight_layout(); plt.savefig(F4, bbox_inches="tight"); plt.close()

# Top nodes bar
F5 = os.path.join(outdir, "fig5_top_nodes.png")
topk = node_totals.head(15)
plt.figure(); plt.bar(topk["node"], topk["total_detections"], color="#8c564b")
plt.xticks(rotation=60, ha="right"); plt.title(f"Top 15 {'States' if use_states else 'Counties'} by Detections"); plt.ylabel("Detections")
plt.tight_layout(); plt.savefig(F5, bbox_inches="tight"); plt.close()

# Display tables
from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("USDA H5N1 — Summary (Plug-in Fit)", summary)
display_dataframe_to_user("USDA H5N1 — Attribution by Week", attr_by_week.head(30))
display_dataframe_to_user(f"Top {'States' if use_states else 'Counties'} by Detections", node_totals.head(25))
display_dataframe_to_user("USDA H5N1 — phi_t (time-varying spread multiplier)", phi_table.head(20))

# Zip all
zip_path = "h5n1_outputs_ultrafast.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(outdir):
        for f in files:
            full = os.path.join(root, f)
            arc = os.path.relpath(full, start=os.path.dirname(outdir))
            zf.write(full, arc)

print("Outputs directory:", outdir)
print("ZIP archive:", zip_path)


In [2]:
# ============================================================
# USDA H5N1 Poultry Detections: Feasible-Only Pipeline
# - If a date-like field exists: state-week endemic–epidemic model
# - If not: cross-sectional summaries only (no dynamics; no cheating)
# - Robust auto-detection of columns + graceful fallbacks
# - Good-looking Matplotlib plots (no seaborn), one chart per figure
# - All outputs zipped + a README explaining what was/wasn't feasible
# ============================================================

import os, io, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------- USER SETTINGS ---------------
CSV_PATH   = "data-table.csv"  # <-- set to your uploaded CSV in Colab
OUTPUT_DIR = "h5n1_outputs_feasible"
ZIP_PATH   = "h5n1_outputs_feasible.zip"

# If the dataset is huge at county-level, cap to top-N counties by total detections
MAX_COUNTIES = 400

# Choose your default modeling scale if time info exists
DEFAULT_LEVEL = "state"   # "state" (fast/stable) or "county"
TIME_SCALE    = "weekly"  # "weekly" (recommended) or "daily"

# Renewal kernel settings (weekly or daily)
RENEWAL_L    = 6  if TIME_SCALE == "weekly" else 14
RENEWAL_MEAN = 2.0 if TIME_SCALE == "weekly" else 7.0
RENEWAL_SD   = 1.0 if TIME_SCALE == "weekly" else 3.0

# Spatial kernel
KERNEL_RHO = 1.5
KERNEL_D0  = 10.0  # km

# Smoother (second-difference penalty) for time-varying phi_t
PHI_SMOOTH_LAMBDA = 50.0

np.random.seed(1)
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 140})

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()

# ----------------- Utility -------------------
def find_col(df, candidates):
    """Return first exact or case-insensitive match from candidates."""
    for c in candidates:
        if c in df.columns: return c
        for col in df.columns:
            if col.lower() == c.lower(): return col
    return None

def try_parse_datetime(s):
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.tz_localize(None)
    return pd.to_datetime(s, errors="coerce").dt.tz_localize(None)

def best_date_column(df):
    """
    Try hard to find/construct a date-like column:
      1) Obvious names: date, report_date, confirmation_date, event_date, detected_on, etc.
      2) Any column where >=30% parses as datetime
      3) Combine year-month-day integer columns
      4) epiweek/year or "YYYY-WXX" strings -> Monday-of-week
    Return: (pd.Series of datetime or None, reason string)
    """
    # 1) Common names
    common = [
        "date","Date","report_date","confirmation_date","Reported Date","event_date",
        "reported_on","confirmed_on","detected_on","timestamp","created_at","updated_at",
        "Report Date","Outbreak Start Date","Outbreak Date","Start Date"
    ]
    col = find_col(df, common)
    if col is not None:
        s = try_parse_datetime(df[col])
        if s.notna().sum() >= max(10, int(0.3*len(df))):
            return s, f"parsed from column '{col}'"

    # 2) Any column with >=30% valid datetimes
    for col in df.columns:
        s = try_parse_datetime(df[col])
        if s.notna().sum() >= max(10, int(0.3*len(df))):
            return s, f"parsed from generic datetime-like column '{col}'"

    # 3) year-month-day combinations
    ycol = find_col(df, ["year","Year","YEAR"])
    mcol = find_col(df, ["month","Month","MONTH"])
    dcol = find_col(df, ["day","Day","DAY","day_of_month"])
    if ycol and mcol and dcol:
        try:
            y = pd.to_numeric(df[ycol], errors="coerce")
            m = pd.to_numeric(df[mcol], errors="coerce")
            d = pd.to_numeric(df[dcol], errors="coerce")
            s = pd.to_datetime(dict(year=y, month=m, day=d), errors="coerce")
            if s.notna().sum() >= max(10, int(0.3*len(df))):
                return s.dt.tz_localize(None), f"constructed from year/month/day ({ycol}/{mcol}/{dcol})"
        except Exception:
            pass

    # 4) epiweek/year OR year-week strings
    wkcol = find_col(df, ["week","Week","epiweek","EpiWeek","iso_week","ISO_Week"])
    ycol2 = find_col(df, ["year","Year","YEAR","fiscal_year"])
    if ycol2 and wkcol:
        try:
            y = pd.to_numeric(df[ycol2], errors="coerce").fillna(1970).astype(int)
            w = pd.to_numeric(df[wkcol], errors="coerce").fillna(1).astype(int)
            # Monday of ISO week:
            s = pd.to_datetime(y.astype(str) + "-W" + w.astype(str) + "-1", format="%G-W%V-%u", errors="coerce")
            if s.notna().sum() >= max(10, int(0.3*len(df))):
                return s.dt.tz_localize(None), f"constructed from year/epiweek ({ycol2}/{wkcol})"
        except Exception:
            pass

    # 4b) year-week strings in a single column
    wkstr = find_col(df, ["year_week","YearWeek","yearweek","yr_wk"])
    if wkstr:
        s = pd.to_datetime(df[wkstr], errors="coerce")
        if s.notna().sum() >= max(10, int(0.3*len(df))):
            return s.dt.tz_localize(None), f"parsed from '{wkstr}'"

    return None, "no suitable date-like field found"

def discrete_gamma_weights(L=14, mean=7.0, sd=3.0):
    from math import gamma as Gamma
    if mean<=0 or sd<=0:
        w = np.zeros(L); w[0]=1.0; return w
    k = (mean/sd)**2
    theta = (sd**2)/mean
    t = np.arange(1, L+1, dtype=float)
    pdf = (t**(k-1))*np.exp(-t/theta) / (Gamma(k) * (theta**k))
    w = pdf / max(pdf.sum(), 1e-12)
    return w

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1 = np.deg2rad(lat1); lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2); lon2 = np.deg2rad(lon2)
    dlat = lat2-lat1; dlon = lon2-lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arcsin(np.minimum(1.0, np.sqrt(a)))
    return R*c

def build_kernel(D, rho=1.5, d0=10.0):
    K = (D + d0)**(-rho)
    rs = K.sum(axis=1, keepdims=True)
    rs[rs==0]=1.0
    return K/rs

def second_difference_matrix(T):
    if T<=2:
        return np.zeros((0, T))
    D = np.zeros((T-2, T))
    for i in range(T-2):
        D[i,i]=1; D[i,i+1]=-2; D[i,i+2]=1
    return D

# ----------------- Load Data ------------------
df = pd.read_csv(CSV_PATH)
orig_cols = df.columns.tolist()

# Detect core columns (state, county, lat/lon, count)
state_col = find_col(df, ["state","State","province_state","Province_State","STATE"])
county_col= find_col(df, ["county","County","county_name","County_Name","COUNTY"])
lat_col   = find_col(df, ["lat","latitude","Lat","Latitude"])
lon_col   = find_col(df, ["lon","long","longitude","Longitude","Long","lng"])
count_col = find_col(df, ["count","detections","n","events","Cases","cases","detections_count","num_events"])

# Build date (if possible)
date_series, date_reason = best_date_column(df)
has_date = date_series is not None
if has_date:
    df["__date__"] = date_series
    df = df.dropna(subset=["__date__"]).copy()

# Clean state/county if present
if state_col is None:
    df["__state__"] = "Unknown"
    state_col = "__state__"
else:
    df[state_col] = df[state_col].astype(str).str.strip()

if county_col is not None:
    df[county_col] = df[county_col].astype(str).str.strip()

# Count: if missing, default to 1 per row (events list)
if count_col is None:
    df["__count__"] = 1
    count_col = "__count__"

# --------- Feasibility report ----------
feas_lines = []
feas_lines.append("Feasibility Report")
feas_lines.append("===================")
feas_lines.append(f"- Date field: {'FOUND' if has_date else 'NOT FOUND'}" + (f" ({date_reason})" if has_date else ""))
feas_lines.append(f"- State field: {'FOUND' if state_col else 'NOT FOUND'}")
feas_lines.append(f"- County field: {'FOUND' if county_col else 'NOT FOUND'}")
feas_lines.append(f"- Lat/Lon fields: {'FOUND' if (lat_col and lon_col) else 'NOT FOUND'}")
feas_lines.append(f"- Count field: '{count_col}' used (1 per row if not provided)")
feas_lines.append("")

# ----------------- If NO date: cross-sectional only ------------------
if not has_date:
    # National totals
    total_events = int(df[count_col].sum())
    # State totals
    state_totals = (df.groupby(state_col)[count_col].sum()
                      .sort_values(ascending=False)
                      .reset_index().rename(columns={count_col: "total_detections"}))

    # County totals (if county exists)
    if county_col:
        df["node"] = df[state_col].astype(str) + "|" + df[county_col].astype(str)
        county_totals = (df.groupby("node")[count_col].sum()
                           .sort_values(ascending=False)
                           .reset_index().rename(columns={count_col:"total_detections"}))
    else:
        county_totals = pd.DataFrame(columns=["node","total_detections"])

    # Save tables
    state_totals.to_csv(os.path.join(OUTPUT_DIR,"state_totals_no_time.csv"), index=False)
    county_totals.to_csv(os.path.join(OUTPUT_DIR,"county_totals_no_time.csv"), index=False)

    # Plots
    # Top 20 states
    top = state_totals.head(20)
    if len(top)>0:
        plt.figure()
        plt.bar(top[state_col], top["total_detections"], color="#1f77b4")
        plt.xticks(rotation=60, ha="right")
        plt.ylabel("Detections (total)")
        plt.title("Top 20 States by Total Detections")
        savefig(os.path.join(OUTPUT_DIR, "fig_top_states_no_time.png"))

    # Top 20 counties (if any)
    topc = county_totals.head(20)
    if len(topc)>0:
        plt.figure()
        plt.bar(topc["node"], topc["total_detections"], color="#d62728")
        plt.xticks(rotation=60, ha="right")
        plt.ylabel("Detections (total)")
        plt.title("Top 20 Counties by Total Detections")
        savefig(os.path.join(OUTPUT_DIR, "fig_top_counties_no_time.png"))

    feas_lines.append("No date-like information was found; time-dynamics (renewal, R_t, FOI decomposition) were NOT attempted.")
    feas_lines.append("What we produced truthfully from this dataset:")
    feas_lines.append("- Cross-sectional national, state, county totals")
    feas_lines.append("- Ranked bar plots of top states/counties")
    feas_lines.append("")
    with open(os.path.join(OUTPUT_DIR, "README.txt"), "w") as f:
        f.write("\n".join(feas_lines))

    # Zip and finish
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(OUTPUT_DIR):
            for ff in files:
                full = os.path.join(root, ff)
                arc = os.path.relpath(full, start=os.path.dirname(OUTPUT_DIR))
                zf.write(full, arc)
    print("No date field found. Cross-sectional outputs saved.")
    print("ZIP:", ZIP_PATH)
    raise SystemExit(0)

# --------------- If date exists: proceed with dynamics ---------------
# Make a modeling "node" column at chosen level
if DEFAULT_LEVEL == "state":
    df["node"] = df[state_col].astype(str)
else:
    if county_col is None:
        warnings.warn("County column not found; falling back to state-level modeling.")
        df["node"] = df[state_col].astype(str)
        DEFAULT_LEVEL = "state"
    else:
        # Limit huge county sets
        totals_node = df.groupby([state_col, county_col])[count_col].sum().reset_index()
        totals_node["node"] = totals_node[state_col].astype(str) + "|" + totals_node[county_col].astype(str)
        if len(totals_node) > MAX_COUNTIES:
            keep_nodes = set(totals_node.sort_values(count_col, ascending=False)
                                           .head(MAX_COUNTIES)["node"])
            df["node_tmp"] = df[state_col].astype(str) + "|" + df[county_col].astype(str)
            df = df[df["node_tmp"].isin(keep_nodes)].copy()
            df["node"] = df["node_tmp"]
            df.drop(columns=["node_tmp"], inplace=True)
        else:
            df["node"] = df[state_col].astype(str) + "|" + df[county_col].astype(str)

# Build time index (weekly or daily)
if TIME_SCALE == "weekly":
    df["timekey"] = df["__date__"].dt.to_period("W").dt.start_time
else:
    df["timekey"] = df["__date__"].dt.normalize()

series = (df.groupby(["node","timekey"])[count_col]
            .sum().reset_index().rename(columns={count_col:"y", "timekey":"time"}))
nodes = sorted(series["node"].unique().tolist())
time_index = pd.date_range(series["time"].min(), series["time"].max(),
                           freq=("W-MON" if TIME_SCALE=="weekly" else "D"))
m, T = len(nodes), len(time_index)
node_to_idx = {n:i for i,n in enumerate(nodes)}
time_to_idx = {t:i for i,t in enumerate(time_index)}

# Y matrix
Y = np.zeros((m, T), dtype=float)
for _, r in series.iterrows():
    i = node_to_idx[r["node"]]
    t = time_to_idx.get(pd.Timestamp(r["time"]))
    if t is not None:
        Y[i, t] += float(r["y"])

# Distance matrix
lat_col_present = lat_col is not None and lon_col is not None and (df[lat_col].dtype.kind in "fiu") and (df[lon_col].dtype.kind in "fiu")
if lat_col_present:
    ll = (df.groupby("node")[[lat_col, lon_col]].mean().reindex(nodes))
    lats = ll[lat_col].values; lons = ll[lon_col].values
    D = np.zeros((m,m), dtype=float)
    for i in range(m):
        D[i,:] = haversine_km(lats[i], lons[i], lats, lons)
else:
    # Fallback distances: if state-level, all moderate; if county-level, within-state closer
    if DEFAULT_LEVEL == "state":
        D = np.full((m,m), 600.0, dtype=float); np.fill_diagonal(D, 0.0)
    else:
        # infer state from "State|County"
        states_of_nodes = [n.split("|")[0] if "|" in n else "UNK" for n in nodes]
        D = np.full((m,m), 800.0, dtype=float)
        for i in range(m):
            for j in range(m):
                if states_of_nodes[i] == states_of_nodes[j]:
                    D[i,j] = 60.0
        np.fill_diagonal(D, 0.0)

K = build_kernel(D, rho=KERNEL_RHO, d0=KERNEL_D0)

# Renewal convolution H (lagged)
def renewal_conv_row(y_row, w):
    # numpy.convolve for stability; strictly causal (shift by 1)
    conv = np.convolve(y_row, w, mode="full")[:len(y_row)]
    return np.concatenate([[0.0], conv[:-1]])

w = discrete_gamma_weights(L=RENEWAL_L, mean=RENEWAL_MEAN, sd=RENEWAL_SD)
H = np.vstack([renewal_conv_row(Y[j,:], w) for j in range(m)])  # (m x T)

# Exposure
E = K @ H
E_nat = E.sum(axis=0)
y_nat = Y.sum(axis=0)

# Endemic baseline nu_t (national) with seasonal basis
doy = pd.DatetimeIndex(time_index).dayofyear.values.astype(float)
omega = 2*np.pi/365.25
X = np.column_stack([np.ones(T), np.sin(omega*doy), np.cos(omega*doy)])

if np.all(y_nat == 0):
    beta = np.array([0.0, 0.0, 0.0])
else:
    beta, *_ = np.linalg.lstsq(X, y_nat, rcond=None)
nu_nat = X @ beta
nu_nat = np.clip(nu_nat, 1e-6, None)

# Allocate endemic to nodes by baseline weights
w_node = Y.mean(axis=1)
ss = w_node.sum()
w_node = (np.ones(m)/m) if ss<=0 else (w_node/ss)
NU = np.outer(w_node, nu_nat)

# Solve for smooth nonnegative phi_t:
# min_phi ||diag(E_nat) phi - (y_nat - nu_nat)||^2 + lambda ||D2 phi||^2, phi>=0
b = y_nat - nu_nat
Ediag = np.diag(E_nat)
ATA = Ediag.T @ Ediag
D2 = second_difference_matrix(T)
pen = PHI_SMOOTH_LAMBDA * (D2.T @ D2) if D2.size else np.zeros((T,T))
lhs = ATA + pen + 1e-6*np.eye(T)
rhs = Ediag.T @ b
phi_t = np.linalg.solve(lhs, rhs)
phi_t = np.clip(phi_t, 0.0, None)

PHI = np.tile(phi_t, (m,1))
LAM = NU + PHI * E
R_t_proxy = phi_t.copy()

# ---------- Outputs ----------
summary = pd.DataFrame([{
    "level": DEFAULT_LEVEL,
    "time_scale": TIME_SCALE,
    "nodes": int(m),
    "time_points": int(T),
    "kernel_rho": KERNEL_RHO,
    "kernel_d0_km": KERNEL_D0,
    "renewal_L": RENEWAL_L,
    "renewal_mean": RENEWAL_MEAN,
    "renewal_sd": RENEWAL_SD,
    "phi_smooth_lambda": PHI_SMOOTH_LAMBDA,
    "total_observed": float(y_nat.sum()),
    "total_fitted": float(LAM.sum())
}])
summary.to_csv(os.path.join(OUTPUT_DIR, "summary.csv"), index=False)

node_totals = pd.DataFrame({"node": nodes, "total_detections": Y.sum(axis=1)}).sort_values("total_detections", ascending=False)
node_totals.to_csv(os.path.join(OUTPUT_DIR, "node_totals.csv"), index=False)

attr_by_time = pd.DataFrame({
    "time": time_index,
    "observed_total": y_nat,
    "expected_total": LAM.sum(axis=0),
    "endemic_expected": NU.sum(axis=0),
    "epidemic_expected": (PHI*E).sum(axis=0),
    "phi_t": phi_t,
    "R_t_proxy": R_t_proxy
})
den = np.maximum(attr_by_time["expected_total"].values, 1e-9)
attr_by_time["AF_endemic"] = attr_by_time["endemic_expected"].values / den
attr_by_time["AF_epidemic"] = 1.0 - attr_by_time["AF_endemic"].values
attr_by_time.to_csv(os.path.join(OUTPUT_DIR, "attribution_by_time.csv"), index=False)

# Metrics (aggregate)
mae = float(np.mean(np.abs(y_nat - LAM.sum(axis=0))))
rmse = float(np.sqrt(np.mean((y_nat - LAM.sum(axis=0))**2)))
pd.DataFrame([{"MAE_total": mae, "RMSE_total": rmse}]).to_csv(os.path.join(OUTPUT_DIR, "metrics.csv"), index=False)

# --------- Plots (skip empties to avoid blank charts) ---------
# Obs vs Fitted (aggregate)
if np.any(y_nat>0) or np.any(LAM.sum(axis=0)>0):
    plt.figure()
    plt.plot(time_index, y_nat, lw=2.0, label="Observed total", color="#1f77b4")
    plt.plot(time_index, LAM.sum(axis=0), lw=2.0, label="Fitted total (NU + PHI*E)", color="#d62728")
    plt.xlabel("Time"); plt.ylabel(f"Detections per {'week' if TIME_SCALE=='weekly' else 'day'}")
    plt.title(f"Observed vs Fitted — {DEFAULT_LEVEL.capitalize()}-{TIME_SCALE}")
    plt.legend()
    savefig(os.path.join(OUTPUT_DIR, "fig_obs_vs_fitted_total.png"))

# Endemic vs Epidemic
end_tot = NU.sum(axis=0); epi_tot = (PHI*E).sum(axis=0)
if np.any(end_tot>0) or np.any(epi_tot>0):
    plt.figure()
    plt.fill_between(time_index, end_tot, color="#17becf", alpha=0.5, label="Endemic expected")
    plt.plot(time_index, epi_tot, lw=2.0, color="#ff7f0e", label="Epidemic expected")
    plt.xlabel("Time"); plt.ylabel("Expected detections")
    plt.title("Endemic vs Epidemic (aggregate)")
    plt.legend()
    savefig(os.path.join(OUTPUT_DIR, "fig_endemic_vs_epidemic_total.png"))

# Rt proxy
if np.any(R_t_proxy>0):
    plt.figure()
    plt.plot(time_index, R_t_proxy, lw=2.0, color="#2ca02c")
    plt.axhline(1.0, lw=1.0)
    plt.xlabel("Time"); plt.ylabel(r"$R_t$ proxy ($\phi_t$)")
    plt.title(r"System-level $R_t$ Proxy from $\phi_t$")
    savefig(os.path.join(OUTPUT_DIR, "fig_Rt_proxy_phi_t.png"))

# Attribution fractions
if np.any(attr_by_time["expected_total"].values>0):
    plt.figure()
    plt.plot(time_index, attr_by_time["AF_endemic"].values, lw=2.0, label="AF_endemic", color="#9467bd")
    plt.plot(time_index, attr_by_time["AF_epidemic"].values, lw=2.0, label="AF_epidemic", color="#8c564b")
    plt.xlabel("Time"); plt.ylabel("Attribution fraction")
    plt.title("Attribution (Endemic vs Epidemic) Fractions")
    plt.legend()
    savefig(os.path.join(OUTPUT_DIR, "fig_attribution_fractions.png"))

# Kernel vs distance (diagnostic)
if np.isfinite(D).all():
    dvals = np.linspace(0, max(1.0, float(np.nanmax(D))), 300)
    kvals = (dvals + KERNEL_D0)**(-KERNEL_RHO)
    kvals = kvals / max(kvals.sum(), 1e-12)
    plt.figure()
    plt.plot(dvals, kvals, lw=2.0, color="#9467bd")
    plt.xlabel("Distance (km)"); plt.ylabel("Scaled kernel weight")
    plt.title("Power-law Kernel vs Distance")
    savefig(os.path.join(OUTPUT_DIR, "fig_kernel_vs_distance.png"))

# Heatmap topN nodes
topN = min(30, m)
order = np.argsort(-Y.sum(axis=1))[:topN]
heat = Y[order,:]
if np.any(heat>0):
    plt.figure()
    plt.imshow(heat, aspect="auto", interpolation="nearest")
    plt.colorbar(label=f"Detections per {'week' if TIME_SCALE=='weekly' else 'day'}")
    yticks = [nodes[i] for i in order]
    plt.yticks(np.arange(len(yticks)), yticks)
    xt = np.linspace(0, T-1, num=min(12, T)).astype(int)
    plt.xticks(xt, [str(pd.Timestamp(time_index[i]).date()) for i in xt], rotation=60, ha="right")
    plt.title(f"{DEFAULT_LEVEL.capitalize()}-{TIME_SCALE} heatmap (Top {topN} nodes)")
    savefig(os.path.join(OUTPUT_DIR, "fig_heatmap_nodes_topN.png"))

# Top nodes bar
top15 = node_totals.head(15)
if len(top15)>0:
    plt.figure()
    plt.bar(top15["node"], top15["total_detections"], color="#8c564b")
    plt.xticks(rotation=60, ha="right")
    plt.ylabel("Total detections")
    plt.title(f"Top 15 {DEFAULT_LEVEL.capitalize()}s by Total Detections")
    savefig(os.path.join(OUTPUT_DIR, "fig_top_nodes_bar.png"))

# Node-wise obs vs fit for top 6
top6 = node_totals.head(6)["node"].tolist()
for nname in top6:
    i = node_to_idx[nname]
    if np.any(Y[i,:]>0) or np.any(LAM[i,:]>0):
        pth = os.path.join(OUTPUT_DIR, f"fig_obs_vs_fit_{nname.replace('|','_')}.png")
        plt.figure()
        plt.plot(time_index, Y[i,:], lw=1.8, label="Observed", color="#1f77b4")
        plt.plot(time_index, LAM[i,:], lw=1.8, label="Fitted", color="#d62728")
        plt.xlabel("Time"); plt.ylabel("Detections")
        plt.title(f"{nname} — Observed vs Fitted")
        plt.legend()
        savefig(pth)

# README
feas_lines.append("Date-like information FOUND — dynamic endemic–epidemic modeling RUN.")
feas_lines.append(f"- Model level: {DEFAULT_LEVEL}")
feas_lines.append(f"- Time scale: {TIME_SCALE}")
feas_lines.append(f"- Renewal kernel: L={RENEWAL_L}, mean={RENEWAL_MEAN}, sd={RENEWAL_SD}")
feas_lines.append(f"- Spatial kernel: rho={KERNEL_RHO}, d0={KERNEL_D0} km (row-stochastic)")
feas_lines.append(f"- phi_t smoothing lambda={PHI_SMOOTH_LAMBDA} (second-difference penalty)")
with open(os.path.join(OUTPUT_DIR, "README.txt"), "w") as f:
    f.write("\n".join(feas_lines))

# ZIP all outputs
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for ff in files:
            full = os.path.join(root, ff)
            arc = os.path.relpath(full, start=os.path.dirname(OUTPUT_DIR))
            zf.write(full, arc)

print("All outputs saved to:", OUTPUT_DIR)
print("ZIP archive created at:", ZIP_PATH)


All outputs saved to: h5n1_outputs_feasible
ZIP archive created at: h5n1_outputs_feasible.zip
